<div align="center">
  <h1></h1>
  <h1>Retrieval-Augmented Generation</h1>
  <h4 align="center">Assignmnet II</h4>
</div>

# Part 1 – Foundational Concepts

Welcome to Assignment II! In this notebook, you will build and implement a Retrieval-Augmented Generation (RAG) pipeline.



### Please answer the following questions in short. Avoid using llms to answer this part!

1. Define RAG and describe how it improves generative model responses compared to an LLM without retrieval.

2. List and briefly explain the core stages of a RAG pipeline (indexing, retrieval, augmentation, generation).


3. Explain why RAG can reduce “hallucinations” compared to plain generative models — and why it doesn’t eliminate them entirely.


4. Compare two retrieval methods (e.g., BM25 vs Dense Embeddings) and discuss how they affect RAG performance.

# Part 2 – Implementation

# 2.1. Access to Groq
Execute the following cell to connect to your Groq account.

In [17]:
import os
os.environ["GROQ_API_KEY"]="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"


# 2.2. Packages  
Execute the following code cells for installing the packages needed.

note: If there are package conflics you can use pip-tools to automatically find and install the compatible versions. If you won't be using specific libraries that can't be installed, you can ignore them.

In [18]:
!pip install -q requests==2.32.4
!pip install -q langchain langchain-community chromadb sentence-transformers python-dotenv


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-community 0.4.1 requires requests<3.0.0,>=2.32.5, but you have requests 2.32.4 which is incompatible.

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [19]:
import requests
import json

from langchain_core.documents import Document

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma


In [20]:
!pip install -q groq langchain-groq beautifulsoup4



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


# 2.3. LLM Interface (Groq API)

This cell defines a simple, reusable interface for interacting with a large language model hosted by Groq. It initializes the Groq client using an API key from the environment and provides a helper function (call_groq_llm) that sends a prompt to a specified Groq model and returns the generated response.

In [21]:
import os
from groq import Groq

groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])

def call_groq_llm(prompt, model="llama-3.1-8b-instant"):
    response = groq_client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt},
        ],
        temperature=0.2,
    )
    return response.choices[0].message.content


# 2.4. Document Preparation and Chunking

In this cell, you will prepare the knowledge base used by the RAG system. Your task is to define a small collection of documents that represent external knowledge and then split these documents into smaller, overlapping chunks suitable for retrieval. Proper document selection and chunking are crucial, as they directly affect retrieval quality and, consequently, the final generated answers.

In [22]:
# TODO:
# 1. Extend the `docs` list with more Document objects related to RAG.
# 2. Make sure the documents contain meaningful explanatory text (not just one sentence).
# 3. Optionally adjust `chunk_size` and `chunk_overlap` and justify your choice.

docs = [
    Document(page_content="Retrieval Augmented Generation (RAG) helps ground generation in external knowledge."),
    Document(page_content="Advanced techniques like Self-RAG and RRR improve answer quality using refinement loops."),
    # TODO: add more documents here
    Document(page_content="A typical RAG pipeline consists of document ingestion, text chunking"),
    Document(page_content="embedding generation, vector storage, retrieval, and response generation."),
    Document(page_content="Documents are split into smaller chunks, embedded into a vector space"),
    Document(
    page_content=("Self-RAG (Self-Reflective Retrieval-Augmented Generation) is an advanced RAG "
        "framework where the language model actively evaluates its own responses and "
        "retrieved evidence. The model can decide when retrieval is necessary, critique "
        "the relevance of retrieved documents, and refine queries iteratively. "
        "This self-reflection mechanism improves factual accuracy and reduces hallucinations "
        "by ensuring that answers are well-supported by external knowledge."
    )
    ),
    Document(
        page_content=(
            "Chunking strategy plays a critical role in RAG performance. If chunks are "
            "too large, irrelevant information may be retrieved and waste context window "
            "space. If chunks are too small, important context may be lost. Overlapping "
            "chunks help preserve semantic continuity across boundaries, improving "
            "retrieval quality for longer explanations or narratives.")),
    Document(
        page_content=(
            "Evaluating RAG systems requires measuring both retrieval quality and generation "
            "quality. Common retrieval metrics include recall@k and MRR, while generation "
            "is evaluated using faithfulness, answer correctness, and groundedness. "
            "Human evaluation is often necessary to detect subtle hallucinations or "
            "missing evidence that automatic metrics may overlook."
        )
    ),
    
]

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
split_docs = splitter.split_documents(docs)


# 2.5. Embedding Creation and Vector Store Indexing

In this cell, you will transform the text chunks created earlier into vector embeddings and store them in a vector database. These embeddings enable semantic search, allowing the system to retrieve relevant pieces of information based on meaning rather than exact word matches. You will also configure how many relevant chunks are retrieved for each query. This step forms the core “retrieval” component of the RAG pipeline.

* Initialize an embedding model suitable for semantic similarity search.

* Create and configure a vector store that indexes the document chunks produced in the previous cell.

* Add the chunked documents to the vector store.

* Set up a retrieval configuration (e.g., number of results k) and consider how this choice might affect answer quality.

In [23]:
embeddings = HuggingFaceEmbeddings(
   # TODO: 1. Initialize an embedding model for semantic search.
    model_name="sentence-transformers/all-MiniLM-l6-v2"
    
)

vectorstore = Chroma(
    # TODO: 2. Create a vector store to index the document chunks.
    collection_name="rag_docs",
    embedding_function=embeddings
)

vectorstore.add_documents(split_docs) # 3. Add the split documents to the vector store.

retriever = vectorstore.as_retriever(
    # TODO: 4. Configure the retriever (e.g., choose an appropriate value for k).
    search_type="similarity",
    search_kwargs={"k":4}
)


# 2.6. Basic RAG

In this cell, you will implement a complete, end-to-end RAG pipeline. Given a user question, the system first retrieves semantically relevant document chunks from the vector store, then constructs a context from these chunks, and finally uses a large language model to generate an answer grounded in the retrieved information. This cell brings together all previous steps—retrieval, context construction, and generation—into a single working function.

In [24]:
def basic_rag_answer(question, k=5):
    # TODO: 1) Retrieve similar documents from the vector store
    retrieved_docs = vectorstore.similarity_search(question,k=k)

    # TODO: 2) Build a single context string from retrieved chunks
    context ="\n\n".join(doc.page_content for doc in retrieved_docs)
    # TODO: 3) Construct a grounded prompt
    prompt = (
        "you are a helpful assistant . Answer the question using ONLY the infomation "
        "provided in the context below. \n\n"
        "Context :\n"
        f"{context}\n\n"
        "Question:\n"
        f"{question}\n\n"
        "If the answer connot be found in the context , say so explicitly."
)

    # 4) Call the LLM to generate the answer
    return call_groq_llm(prompt)


In [25]:
print(basic_rag_answer("What is Self-RAG?"))


Self-RAG is an advanced RAG framework where the language model actively evaluates its own responses and retrieved evidence.


# 2.7. End-to-End RAGPipeline with Web Data

In this cell, you will construct a complete RAG pipeline using real-world web data.
The pipeline performs the following steps:

* Document loading from a live webpage

* Text chunking for efficient retrieval

* Embedding and vector storage using a dense vector database

* Retrieval of relevant document chunks for a query

* Grounded generation using a large language model

This cell demonstrates how RAG systems are engineered in practice, from raw data ingestion to final answer generation. Please implement the cells as instructed or write in the markdown for answering if needed.

In [26]:
# TODO:
# - Read through the imports and identify which ones are responsible for:

import bs4
import os
#   (2) text chunking
from langchain_text_splitters import RecursiveCharacterTextSplitter
#   (1) document loading
from langchain_community.document_loaders import WebBaseLoader
#   (4) retrieval
from langchain_community.vectorstores import Chroma
#   (3) embeddings
from langchain_community.embeddings import HuggingFaceEmbeddings
#   (5) generation
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

from langchain_groq import ChatGroq


USER_AGENT environment variable not set, consider setting it to identify your requests.


In [ ]:
# TODO:
# - Replace the URL with another relevant article about RAG, agents, or LLM reasoning
# - (Optional) Add a second URL to compare retrieval behavior

loader = WebBaseLoader(
    web_paths=(
        "https://blogs.nvidia.com/blog/what-is-retrieval-augmented-generation/",
        # TODO: add another URL here
        "https://www.mckinsey.com/featured-insights/mckinsey-explainers/what-is-retrieval-augmented-generation-rag"
    ),
)
docs = loader.load()

# TODO:
# - Print the number of loaded documents
print(f"Number of loaded documents: {len(docs)}")

# - Inspect the content of one document
for i, doc in enumerate(docs, start=1):
    print(f"Document {i} length:", len(doc.page_content))
if docs:
    # print first 5 lines
    lines = [line.strip() for line in docs[0].page_content.splitlines() if line.strip()]
    preview_lines = lines[:5]
    for i, line in enumerate(preview_lines, start=1):
        print(f"{i}: {line}")

In [ ]:
# TODO:
# - Experiment with different chunk_size and chunk_overlap values
# - Explain (in a markdown cell) why chunking is necessary for RAG

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,      # TODO: try 500 or 1500
    chunk_overlap=50     # TODO: try 50 or 300
)

splits = text_splitter.split_documents(docs)

# TODO:
# - Print the number of chunks created
print(f"Number of chunks created: {len(splits)}")
# - Inspect the length of one chunk
if splits:
    first_chunk = splits[0].page_content
    print("\nFirst chunk preview (first 800 chars):\n")
    print(first_chunk[:800])
    print("\nLength of first chunk:", len(first_chunk))


Number of chunks created: 131

First chunk preview (first 500 chars):

RAG at the Crossroads - Mid-2025 Reflections on AI’s Incremental Evolution | RAGFlow

Length of first chunk: 84


# Why Chunking is Necessary for RAG  

retrieval Augmented Generation relies on retrieving relevant pieces of information from a large corpus / Data facts . Using themas a context for a language model to genrate answers.
* Chunking is crutial preprocessing several other reasons
* Fitting LLM context limits.
* small chunks helps the systems to find the most relevant information.
* overlapping chunks keep important sentences intact.
* Better scales as the efficient retrieval from large document collections.


In [ ]:
# TODO:
# - Read about the embedding model used below
# - Explain why dense embeddings enable semantic search

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings
)

# TODO:
# - Change k and observe how retrieval results differ
retriever = vectorstore.as_retriever(search_kwargs={"k": 8})


In [ ]:
# TODO:
# - Modify the prompt to further reduce hallucinations
# - Add instructions such as:
#   “If the answer is not in the context, say 'I don't know'.”

prompt = ChatPromptTemplate.from_template(
    """You are a helpful and factual assistant.

You MUST answer the question using ONLY the information provided in the context below.
Do NOT use any external knowledge or assumptions.

If the answer cannot be found in the context, respond exactly with:
"I don't know."

Context:
{context}

Question:
{question}

Answer:
"""
)


In [ ]:
# TODO:
# - Try a different Groq model
# - Experiment with temperature > 0 and observe changes

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.3
)


In [ ]:
# TODO:
# - Explain why retrieved documents must be formatted before prompting
# - (Optional) Add separators or document numbering

def format_docs(docs):
    formatted_docs = []
    for i, doc in enumerate(docs, start=1):
        formatted_docs.append(f"[Document {i}]\n{doc.page_content}")
        return "\n\n---\n\n".join(formatted_docs)


In [ ]:
# TODO:
# - Trace the data flow in this chain:
#   question → retrieval → formatting → prompt → LLM → output
# - Identify which component is responsible for retrieval

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)


In [ ]:
# TODO:
# - Ask at least two different questions
# - Compare answers with different k values
# - Identify one failure case where RAG performs poorly
question_1= "How does RAG support an agent’s memory system according to the passage?"
question_2= "Why does data quality degrade quickly in centralized RAG repositories? "

answer_1=rag_chain.invoke(question_1)
answer_2=rag_chain.invoke(question_2)

print("Q1 :",question_1)
print("A1:",answer_1,"\n")

print("Q2:",question_2)
print("A2:",answer_2)
#rag_chain.invoke("What is Task Decomposition?")


Q1 : How does RAG support an agent’s memory system according to the passage?
A1: RAG builds the agent's long-term memory, enabling task state tracking and context acceleration through indexing, forgetting, and consolidation, while working with short-term memory to form a complete architecture. 

Q2: Why does data quality degrade quickly in centralized RAG repositories? 
A2: Data quality degrades quickly as information in these repositories becomes stale, requiring constant synchronization with source systems.


# 2.8. RRR-RAG (Rewrite–Retrieve–Respond)

This cell implements RRR-RAG (Rewrite–Retrieve–Respond), an extension of basic RAG.
Instead of directly retrieving documents using the user’s original question, the system first rewrites the question to make it more suitable for semantic search. The rewritten query is then used to retrieve relevant documents from the vector store, and finally the model generates an answer grounded only in the retrieved context.

The goal of this approach is to improve retrieval quality and, as a result, produce more accurate and relevant answers compared to standard RAG.

In [ ]:
def rrr_rag_answer(question):
    """
    Rewrite–Retrieve–Respond RAG

    This function improves retrieval quality by first rewriting
    the user question, then retrieving documents using the rewritten
    query, and finally generating an answer grounded in the retrieved context.
    """

    # TODO 1:
    # Write a prompt that rewrites the input question to be more retrieval-friendly.
    # The prompt should:
    # - Clearly instruct the model to rewrite the question
    # - Include the original question
    rewrite_prompt = f"""
    You are an assistant that rewrites questions to improve semantic search.
    Rewrite the following question to make it more precise and suitable for document retrieval,
    without changing its meaning.

    
    Original Question: {question}
    
    Rewritten Question:
    """

    # TODO 2:
    # Call the Groq LLM to generate the rewritten query
    q_rewrite = llm.invoke(rewrite_prompt).content

    # TODO 3:
    # Retrieve relevant documents using the rewritten query
    # Hint: use vectorstore.similarity_search(...)
    docs = vectorstore.similarity_search(q_rewrite, k=5) 

    # TODO 4:
    # Build a single context string from the retrieved documents
    ctx = format_docs(docs)

    # TODO 5:
    # Construct the final answer prompt using:
    # - the retrieved context
    # - the original question (NOT the rewritten one)
    answer_prompt = f"""
    You are a helpful assistant that answers questions using ONLY the provided context.
    Do NOT hallucinate; if the answer is not in the context, respond with "I don't know."

    Context:
    {ctx}

    Question:
    {question}

    Answer:
    """
    # TODO 6:
    # Call the Groq LLM to generate the final answer
    return llm.invoke(answer_prompt)


In [ ]:
print(rrr_rag_answer("Explain how refinement can improve RAG quality."))


content="I don't know." additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 6, 'prompt_tokens': 208, 'total_tokens': 214, 'completion_time': 0.004211136, 'completion_tokens_details': None, 'prompt_time': 0.013364429, 'prompt_tokens_details': None, 'queue_time': 0.15352915, 'total_time': 0.017575565}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_020e283281', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None} id='run--2e252d12-c936-4e06-b654-c4bb3b6d134c-0' usage_metadata={'input_tokens': 208, 'output_tokens': 6, 'total_tokens': 214}


# 2.9. Self-RAG

This cell implements Self-RAG, an iterative variant of Retrieval-Augmented Generation.
Instead of answering the question only once, the system repeatedly retrieves documents, generates an answer, and then refines the query based on the previous answer. Each iteration aims to improve retrieval quality, allowing the model to correct or enrich its understanding over time. This approach demonstrates how feedback loops can be used to improve RAG performance without fine-tuning the model.

In [ ]:
def self_rag(question, iterations=2):
    """
    Self-RAG: Iterative Retrieval-Augmented Generation

    The system repeatedly:
    1) retrieves documents,
    2) generates an answer,
    3) refines the query based on the previous answer.
    """

    # TODO 1:
    # Initialize the query used for the first retrieval step
    current_query = question

    # TODO 2:
    # Initialize a variable to store the model's answer
    answer = ""

    for i in range(iterations):

        # TODO 3: Retrieval
        # Retrieve relevant documents using the current query
        docs = vectorstore.similarity_search(current_query,k=5)

        # TODO 4: Context construction
        # Combine retrieved documents into a single context string
        ctx = format_docs(docs)

        # TODO 5: Answer generation
        # Build a prompt that uses ONLY the retrieved context
        prompt = f"""
        You are a helpful assistant that answers questions using ONLY the provided context.
        Do NOT hallucinate; if the answer is not in the context, respond with "I don't know."

        Context:
        {ctx}

        Question:
        {question}

        Answer:
        """

        # TODO 6:
        # Call the LLM to generate an answer
        answer = llm.invoke(prompt).content

        # TODO 7: Query refinement
        # Write a prompt that refines the current query
        # based on the previous answer to improve retrieval
        refine_prompt = f"""
        You are a system that rewrites questions to improve retrieval.
        Based on the previous answer, rewrite the query to retrieve more relevant documents.
        Original Question: {question}
        Previous Answer: {answer}

        Refined Query:
        """

        # TODO 8:
        # Generate the refined query using the LLM
        refined_msg = llm.invoke(refine_prompt)
        current_query = refined_msg.content.strip()

    # TODO 9:
    # Return the final answer after all iterations
    return answer


In [ ]:
print(self_rag("How does RAG refine answers?", iterations=3))


I don't know.


# 2.10. Comparison of RAG strategies

This cell compares three RAG strategies—Basic RAG, RRR-RAG (Rewrite–Retrieve–Respond), and Self-RAG—on the same set of questions. By running identical queries through different pipelines, you can observe how query rewriting and iterative refinement influence retrieval quality and the final generated answers. This comparison highlights the practical trade-offs between simplicity, robustness, and computational cost in RAG system design.

In [ ]:
questions = [
    "What is the idea of refinement loops in RAG?",
    "How does rewriting improve retrieval?"
]

for q in questions:
    print("----")
    print("Basic RAG    :", basic_rag_answer(q))
    print("RRR RAG      :", rrr_rag_answer(q))
    print("Self-RAG     :", self_rag(q))


----
Basic RAG    : The idea of refinement loops in RAG is not explicitly mentioned in the context. However, it does mention "manual or model-driven reflective loops" which Agents use to tackle RAG reasoning challenges and enable intelligent breakthroughs.
RRR RAG      : content="I don't know." additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 6, 'prompt_tokens': 191, 'total_tokens': 197, 'completion_time': 0.005340752, 'completion_tokens_details': None, 'prompt_time': 0.015063066, 'prompt_tokens_details': None, 'queue_time': 0.090561286, 'total_time': 0.020403818}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None} id='run--f8b43097-ed72-441a-926a-b5da90b3563e-0' usage_metadata={'input_tokens': 191, 'output_tokens': 6, 'total_tokens': 197}
Self-RAG     : I don't know.
----
Basic RAG    : The context does not mention how rewriting improves retrieval.
RRR RAG     

***What You Should Remember:***

1. RAG combines information retrieval with language models so that answers are grounded in external knowledge rather than relying only on the model’s internal memory. This reduces hallucinations and improves factual accuracy.

2. Chroma Vector Store enables semantic retrieval by embedding text chunks into high-dimensional vector spaces. Queries are embedded in the same space, and the most semantically similar chunks are retrieved to provide relevant context.

3. BM25 Retriever performs lexical (keyword-based) retrieval, ranking documents by term frequency and inverse document frequency. It is especially effective when exact wording or terminology matters.

4. High-quality RAG systems often benefit from hybrid retrieval, where:

*  BM25 ensures precision through keyword matching, and

*  Vector retrieval (Chroma) ensures recall through semantic similarity.

5. Building a complete RAG pipeline requires:

*  Data preparation:
Collecting raw text (e.g., web pages, PDFs) and splitting it into manageable chunks that can be efficiently retrieved.

*  Retriever setup:
Creating one or more retrievers (vector-based, lexical, or hybrid) to fetch the most relevant context for a given query.

*  Prompt and context formatting:
Structuring retrieved documents into a clear context block that guides the language model to use only the provided information.

*  LLM integration:
Connecting the retriever output to a language model (e.g., Groq LLMs) that generates the final answer.

6. Advanced RAG techniques improve quality further:

*  RRR-RAG rewrites queries to improve retrieval quality before answering.

*  Self-RAG iteratively refines queries and answers using feedback loops.

7. There is a trade-off between simplicity and performance:

*  Basic RAG is fast and cost-effective.

*  Refinement-based RAG (RRR, Self-RAG) improves answer quality but requires additional LLM calls.

Congratulations! You've come to the end of this assignment.